### Importing necessary libraries

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import requests
from http.cookiejar import MozillaCookieJar
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

### trying llama model

In [ ]:
# model = "llama3.2"
# base_url = "http://localhost:11434"

# llm = ChatOllama(
#     model=model,
#     base_url=base_url,
#     temperature=0.2
# )

##### The ChatOllama model are not that good. The model was unnecessarily hallucinating so i am switching to ChatGroq model with llama-3.3-70b-versatile which have over 70 billion parameters

### Trying groq model

In [ ]:
load_dotenv()
llm = ChatGroq(model_name="llama-3.3-70b-versatile", temperature=0.2)

embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

### Loading Transcript

In [ ]:
def get_transcript(vid_id: str, cookie_file: str = "youtube_cookies.txt") -> str:
    session = requests.Session()
    session.headers.update({
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    })
    cookie_jar = MozillaCookieJar(cookie_file)
    cookie_jar.load(ignore_discard=True, ignore_expires=True)
    session.cookies = cookie_jar

    yt_api = YouTubeTranscriptApi(http_client=session)
    transcript_list = yt_api.fetch(vid_id, languages=["en"])
    return " ".join(chunk.text for chunk in transcript_list)

In [ ]:
def build_retriever(transcript: str):
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    chunks = splitter.create_documents([transcript])
    vector_store = FAISS.from_documents(documents=chunks, embedding=embedding)
    return vector_store.as_retriever(search_type = 'similarity', search_kwargs = {'k':4})

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


In [ ]:


parser = StrOutputParser()

In [ ]:

prompt = PromptTemplate(
        template="""
    You are a helpful assistant that answers questions strictly based on the YouTube video transcript provided.
    If the answer is not found in the transcript, respond with: "I don't know — this topic isn't covered in the video."
    Do not make up or infer anything beyond what is in the transcript.

    Transcript Context:
    {context}

    Question: {question}

    Answer:""",
        input_variables=['context', 'question']
    )

In [ ]:
def main():
    vid_id = str(input("Enter your video Id : "))
    transcript = get_transcript(vid_id)
    if transcript:
        vec = build_retriever(transcript)
        while True:
            question = str(input("Write your Question : "))
            if question.lower() == 'exit':
                print("Bye Bye!")
                break
            else:
                qa_chain = (
                    {
                        "context": RunnableLambda(lambda x: vec.invoke(question)) | RunnableLambda(format_docs),
                        "question": RunnablePassthrough()
                    }
                    | prompt
                    | llm
                    | parser
                )
                answer = qa_chain.invoke(question)
                print(answer) 
    else:
        print("Transcript not found!")
               

main()


In [ ]:
## jPpsYzlowWU
